# Task 6 — Five-phase replay check

The table below uses five verifier snapshots: baseline, exact replay, modified
revision, exact modified replay, and an idle Spark restart.


In [1]:
from pathlib import Path
import json
from IPython.display import display

def find_evidence():
    start = Path.cwd().resolve()
    for root in (start, *start.parents):
        candidate = root / 'docs' / 'evidence' / 'live_pipeline_summary.json'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('docs/evidence/live_pipeline_summary.json not found')

evidence_path = find_evidence()
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
task = evidence['task6']
display({
    'captured_at': evidence['captured_at'],
    'target': task['target'],
    'phases': task['phases'],
    'acceptance_report': task['acceptance_report'],
})

{'captured_at': '2026-07-25T03:30:43.726347Z',
 'target': {'file_path': 'src/lerobot/__init__.py',
  'file_id': 'file_4bcb0fb8af5f208dad26ab6d584c0d2a8e7c995ea7954b70e1d1976ce286f236',
  'baseline_hash': '9de5fe33e0bf693e86e9bf55360942385a504a1baff224577a405ca91ea33838',
  'modified_hash': '6c0a72b26999ff1fbaa6ba5a6a074f86e7b1e9490320093885335ebaae92f4b7'},
 'phases': [{'phase': 'baseline',
   'nodes': 58,
   'distinct_node_ids': 58,
   'edges': 60,
   'distinct_edge_ids': 60,
   'duplicate_node_groups': 0,
   'duplicate_edge_groups': 0,
   'mongo_documents': 1,
   'checkpoint_batch': 121,
   'committed_offset_total': 497},
  {'phase': 'exact_baseline_replay',
   'nodes': 58,
   'distinct_node_ids': 58,
   'edges': 60,
   'distinct_edge_ids': 60,
   'duplicate_node_groups': 0,
   'duplicate_edge_groups': 0,
   'mongo_documents': 1,
   'checkpoint_batch': 122,
   'committed_offset_total': 498},
  {'phase': 'modified',
   'nodes': 81,
   'distinct_node_ids': 81,
   'edges': 88,
   'disti

In [2]:
phases = {row['phase']: row for row in task['phases']}
expected_order = [
    'baseline', 'exact_baseline_replay', 'modified',
    'exact_modified_replay', 'idle_spark_restart',
]
assert [row['phase'] for row in task['phases']] == expected_order
assert task['target']['baseline_hash'] != task['target']['modified_hash']
for row in task['phases']:
    assert row['nodes'] == row['distinct_node_ids']
    assert row['edges'] == row['distinct_edge_ids']
    assert row['duplicate_node_groups'] == row['duplicate_edge_groups'] == 0
    assert row['mongo_documents'] == 1
assert (phases['baseline']['nodes'], phases['baseline']['edges']) == (58, 60)
assert (phases['exact_baseline_replay']['nodes'], phases['exact_baseline_replay']['edges']) == (58, 60)
assert (phases['modified']['nodes'], phases['modified']['edges']) == (81, 88)
assert (phases['exact_modified_replay']['nodes'], phases['exact_modified_replay']['edges']) == (81, 88)
assert phases['idle_spark_restart']['checkpoint_batch'] == phases['exact_modified_replay']['checkpoint_batch'] == 124
assert phases['idle_spark_restart']['committed_offset_total'] == phases['exact_modified_replay']['committed_offset_total'] == 500
report = task['acceptance_report']
assert report['passed'] is True
assert report['check_count'] == 89
assert report['failure_count'] == 0 and report['failures'] == []
display({
    'status': 'PASS',
    'checks_passed': f"{report['check_count']} / {report['check_count']}",
    'failures': report['failure_count'],
    'baseline_graph': {'nodes': 58, 'edges': 60},
    'modified_graph': {'nodes': 81, 'edges': 88},
    'restart_kept_batch_and_offset': {'batch': 124, 'offset_total': 500},
})

{'status': 'PASS',
 'checks_passed': '89 / 89',
 'failures': 0,
 'baseline_graph': {'nodes': 58, 'edges': 60},
 'modified_graph': {'nodes': 81, 'edges': 88},
 'restart_kept_batch_and_offset': {'batch': 124, 'offset_total': 500}}

## Reflection

Exact replay checks duplicate safety. The changed-hash phase checks update and stale-state reconciliation, and the idle restart checks checkpoint recovery without adding a new input event.